# Universal Context Engine (Converged Edition)

**Copyright 2026, Denis Rothman**

**Goal:** In this notebook we will integrate **Oracle 23ai (Converged Enterprise DB):** agents in a universal content engine to prove that the exact same reasoning architecture (engine.py) can seamlessly switch between:

*  Oracle 23ai (Vector Store): For Marketing & Legal use cases.
*  Oracle 23ai (Relational Table): For HR & Recruitment use cases.


# 1.Installation & Setup

In [1]:
#@title Retrieving GitHub Private token (this cell will be deleted when the repository is public)
import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    pt = userdata.get("GITHUB_TOKEN")
    pt=pt.strip()
    if not pt:
        raise userdata.SecretNotFoundError("GITHUB_TOKEN not found.")

    print("GITHUB_TOKENAPI private token loaded successfully.")

except userdata.SecretNotFoundError:
    print('Secret "GITHUB_TOKEN" not found.')
    print('Please activate or add your GITHUB_TOKEN private token to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the GITHUB_TOKEN: {e}")

GITHUB_TOKENAPI private token loaded successfully.


In [2]:
# Install BOTH Pinecone and Oracle dependencies
!pip install oracledb==3.4.1 openai==2.14.0 tiktoken==0.7.0 tqdm==4.67.1 tenacity --quiet
print("✅ Universal Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 47.1 MB/s eta 0:00:00
✅ Universal Dependencies installed.


In [3]:
import os
import sys
from google.colab import drive, userdata
from openai import OpenAI

# Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Initialize Oracle Client

try:
    # Explicitly check for the mandatory OpenAI key
    openai_key = userdata.get("API_KEY")
    if not openai_key:
        raise ValueError("OpenAI API_KEY not found in Secrets.")

    os.environ["OPENAI_API_KEY"] = openai_key
    client = OpenAI()
    print("✅ OpenAI initialized (Mandatory).")

except Exception as e:
    print(f"❌ Critical Error: OpenAI initialization failed. {e}")
    # We raise the error here because OpenAI is required for the Planner to work at all
    raise e

✅ OpenAI initialized (Mandatory).


In [5]:
#@title Retrieving engine library
import time

!curl -L -H "Authorization: token {pt}" -H "Accept: application/vnd.github.v3.raw" "https://api.github.com/repos/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/contents/commons/engine/engine.zip" -o engine.zip

time.sleep(10)
!unzip -o engine.zip -d /content

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13119  100 13119    0     0  50845      0 --:--:-- --:--:-- --:--:-- 50848
Archive:  engine.zip
  inflating: /content/agent_archivist.py  
  inflating: /content/agent_recruiter.py  
  inflating: /content/agents.py      
  inflating: /content/engine.py      
  inflating: /content/helpers.py     
  inflating: /content/oracle_lib.py  
  inflating: /content/registry.py    


In [6]:
# 3. Import Local Modules (Uploaded Files)
try:
    from engine import context_engine
    from registry import AGENT_TOOLKIT
    from oracle_lib import OracleManager
    print("✅ Universal Engine Modules imported.")
except ImportError as e:
    print(f"❌ Module Import Failed: {e}\nUpload engine.py, registry.py, agents.py, helpers.py, oracle_lib.py, agent_archivist.py, agent_recruiter.py")

✅ Universal Engine Modules imported.


In [7]:
# Initialize Oracle Connection (Enterprise Layer)
try:
    OracleManager.initialize()
    print("✅ Oracle 23ai Connection established.")
except Exception as e:
    print(f"⚠️ Oracle Connection Failed: {e}\n(Ignore if you only want to run Pinecone scenarios)")

✅ Oracle 23ai Connection established.


# 2.Initialize the Universal Engine

In [8]:
# 2. Initialize the Universal Engine Wrapper
from engine import context_engine

# We define a wrapper function that pre-configures the engine with our clients and models.
# This allows us to just pass the 'goal' in the scenario steps below.
def run_universal_engine(goal):
    return context_engine(
        goal=goal,
        client=client,
        # [FIX] Removed Pinecone Client (pc), Index, and Namespaces
        generation_model="gpt-5.1",
        embedding_model="text-embedding-3-small"
    )

print("🚀 Universal Context Engine Wrapper is READY.")
# Access the registry dictionary keys directly to see the list of agents
print(list(AGENT_TOOLKIT.registry.keys()))

🚀 Universal Context Engine Wrapper is READY.
['Writer', 'Summarizer', 'OracleArchivist', 'OracleRecruiter']


# 3.Recruitment Scenarios
This execution plan switches to the `OracleRecruiter` agent. The Engine doesn't change code, only the configuration.

In [9]:
#@title Oracle data verification
from oracle_lib import OracleManager

print("\n=== Oracle Hybrid Table Summary ===\n")

# Use the Manager's context manager instead of a raw 'connection' variable
with OracleManager.get_cursor() as cursor:
    # --- 1. Check HR Tables ---
    tables = ['CANDIDATE_POOL', 'RECRUITMENT_RULES']

    for t in tables:
        try:
            cursor.execute(f"SELECT COUNT(*) FROM {t}")
            result = cursor.fetchone()
            if result:
                print(f"Table {t}: {result[0]} rows")
        except Exception as e:
            print(f"Table {t}: Not Found or Error ({e})")

    print("\n--- Sample Candidate (Structured + Vector) ---")
    try:
        cursor.execute("""
            SELECT candidate_id, full_name, salary_expectation,
                   CASE WHEN resume_vector IS NOT NULL THEN '✅ Vector Present' ELSE '❌ No Vector' END
            FROM candidate_pool
            FETCH FIRST 2 ROWS ONLY
        """)
        for row in cursor.fetchall():
            print(row)
    except Exception as e:
        print(f"Error fetching candidates: {e}")

print("\n=== Verification Complete ===")


=== Oracle Hybrid Table Summary ===

Table CANDIDATE_POOL: 5 rows
Table RECRUITMENT_RULES: 3 rows

--- Sample Candidate (Structured + Vector) ---
('CAND_003', 'Casey M.', 210000, '✅ Vector Present')
('CAND_001', 'Alex V.', 165000, '✅ Vector Present')

=== Verification Complete ===


In [10]:
import textwrap

def run_oracle_recruitment_scenario(role: str, width: int = 80):
    """
    Encapsulates Scenario A: Recruitment using the OracleRecruiter agent.

    Args:
        role (str): The job title or role description to search for.
        width (int): The max line length for the output display (default 80).

    Returns:
        tuple: (result, trace)
    """
    # 1. Define the simplified goal based only on the role
    recruitment_goal = role

    # 2. Visual separation for the start
    print("-" * width)
    print(f"Goal: {recruitment_goal}")
    print("-" * width)

    try:
        # 3. Execute the engine
        # Assuming run_universal_engine is defined in global scope
        result, trace = run_universal_engine(recruitment_goal)

        # 4. Present the text results nicely
        print("\n--- FINAL RECRUITMENT EMAILS ---\n")

        # Split by lines to preserve paragraph structure, then wrap lines individually
        if result:
            for line in str(result).splitlines():
                if line.strip(): # If line is not empty
                    print(textwrap.fill(line, width=width))
                else:
                    print() # Preserve empty lines
        else:
            print("No result returned.")

        print("-" * width)
        return result, trace

    except NameError:
        print("Error: `run_universal_engine` is not defined.")
        return None, None

In [11]:
#@title Scenario A : Searching for experienced Python developers
# The text will now automatically wrap at 80 characters for better readability
res, log = run_oracle_recruitment_scenario("Find Experienced Python Developers with experience")

--------------------------------------------------------------------------------
Goal: Find Experienced Python Developers with experience
--------------------------------------------------------------------------------

--- FINAL RECRUITMENT EMAILS ---

**1. Talent Pool Overview**

The current talent pool for Python-focused roles is very limited. Only one
candidate, Quinn R., has a strong Python background. However, Quinn is now
primarily focused on engineering management and leadership rather than hands-on
development. The remaining candidates have little to no meaningful Python
experience and are oriented toward non-Python tech stacks or non-engineering
roles.

---

**2. Shortlist of Top Candidates**

- **Quinn R.**
  - **Years of Experience:** 8
  - **Current Role:** Engineering Manager
  - **Core Python Skills:**
    - Historically strong Python background
    - Capable of technical leadership on Python projects
    - Currently more focused on management than day-to-day coding
  - 

In [17]:
#@ Scenario B Searching for experienced Python developers and writing an email
# The text will now automatically wrap at 80 characters for better readability
res, log = run_oracle_recruitment_scenario("Find Experienced Python Developers with experience and 250000 salary max and write a job interview email")

--------------------------------------------------------------------------------
Goal: Find Experienced Python Developers with experience and 250000 salary max and write a job interview email
--------------------------------------------------------------------------------


ERROR:root:[OracleRecruiter] Failed: OracleManager is not initialized. Call initialize() first.
ERROR:root:[Writer] An error occurred: Writer requires 'facts' (or retrieved_context) or 'previous_content'.
ERROR:root:--- Executor: FATAL ERROR --- Execution failed at step 2 (Writer): Writer requires 'facts' (or retrieved_context) or 'previous_content'.



--- FINAL RECRUITMENT EMAILS ---

No result returned.
--------------------------------------------------------------------------------


In [13]:
#@title Scenario B: Recruitment (Powered by Oracle)
# This execution plan switches to the `OracleRecruiter` agent.
# The Engine doesn't change code, only the configuration.

# We state the constraints clearly in the goal so the Planner passes them to the OracleRecruiter.
recruitment_goal = "Find Experienced Python Developers with a maxi salary of 200000 and experienced in Python services."

print(f"Goal: {recruitment_goal}")

# Execute the engine (The Planner will automatically select OracleRecruiter -> Writer)
result, trace = run_universal_engine(recruitment_goal)

print("\n--- FINAL RECRUITMENT EMAILS ---")
print(result)

Goal: Find Experienced Python Developers with a maxi salary of 200000 and experienced in Python services.

--- FINAL RECRUITMENT EMAILS ---
{'candidate_list': [{'id': 'CAND_004', 'name': 'Riley S.', 'experience': 5, 'salary': 140000, 'match_score': -0.4721769972538, 'summary': 'Backend Java Developer focused on banking transaction systems. Solid experience with Oracle Database, writing complex PL/SQL stored procedures and triggers. Reliable, detail-oriented, and prefers working on backend logic rather than UI.'}, {'id': 'CAND_005', 'name': 'Quinn R.', 'experience': 8, 'salary': 155000, 'match_score': -0.5456285798987753, 'summary': 'Engineering Manager who loves building company culture. Technologist turned leader. Strong Python background but now focuses on team growth, mentorship, and aligning engineering goals with business strategy. Excellent communicator.'}, {'id': 'CAND_001', 'name': 'Alex V.', 'experience': 12, 'salary': 165000, 'match_score': -0.648857765595557, 'summary': 'Sen

In [14]:
#@title Query powered by Oracle
# 4. Scenario B: Recruitment (Powered by Oracle)
# This execution plan switches to the `OracleRecruiter` agent.
# The Engine doesn't change code, only the configuration.

# We state the constraints clearly in the goal so the Planner passes them to the OracleRecruiter.
recruitment_goal = "Find Experienced Python Developers with max salary 200000 and min 3 years experience, then draft a polite rejection email for them mentioning we will keep their resume."

print(f"Goal: {recruitment_goal}")

# Execute the engine (The Planner will automatically select OracleRecruiter -> Writer)
result, trace = run_universal_engine(recruitment_goal)

print("\n--- FINAL RECRUITMENT EMAILS ---")
print(result)

Goal: Find Experienced Python Developers with max salary 200000 and min 3 years experience, then draft a polite rejection email for them mentioning we will keep their resume.

--- FINAL RECRUITMENT EMAILS ---
Subject: Your Application Status

Dear [Candidate Name],

Thank you for your interest in the [Position Title] role and for the time you invested in our application and interview process. We appreciate the effort you put into sharing your experience and background with us.

After careful consideration, we have decided to move forward with other candidates for this position. This was not an easy decision, as we reviewed many strong profiles.

With your permission, we will keep your resume on file for consideration should future opportunities arise that align with your skills and experience. While we cannot make any guarantees about future roles, we would be glad to revisit your application if a suitable opening becomes available.

Thank you again for your time and interest. We wish 

In [15]:
# 4. Scenario B: Recruitment (Powered by Oracle)
# This execution plan switches to the `OracleRecruiter` agent.
# The Engine doesn't change code, only the configuration.

# We state the constraints clearly in the goal so the Planner passes them to the OracleRecruiter.
# UPGRADE: Explicitly used "140000" (integer format) to ensure Planner parsing accuracy.
recruitment_goal = "Find Experienced Python Developers with max salary 200000 and min 3 years experience, then draft a polite rejection email for them mentioning we will keep their resume."

print(f"Goal: {recruitment_goal}")

# Execute the engine (The Planner will automatically select OracleRecruiter -> Writer)
result, trace = run_universal_engine(recruitment_goal)

# --- RESULT HANDLING & DEBUGGING ---
if result:
    print("\n--- FINAL RECRUITMENT EMAILS ---")
    print(result)
else:
    print("\n⚠️ No Final Result returned by Engine. Inspecting Trace...")
    print(f"Trace Status: {trace.status}")

    # 1. Did we get a plan?
    if trace.plan:
        print(f"Plan Generated: {len(trace.plan)} steps")
        for step in trace.plan:
            print(f" - Step {step.get('step')}: {step.get('agent')} -> {step.get('input')}")
    else:
        print("❌ CRITICAL: No Plan was generated. Check your LLM / Planner logic.")

    # 2. Did we execute steps?
    if trace.steps:
        print(f"\nSteps Executed: {len(trace.steps)}")
        last_step = trace.steps[-1]
        print(f"Last Agent: {last_step['agent']}")
        print(f"Last Output Snippet: {str(last_step['output'])[:200]}...")
    else:
        print("❌ CRITICAL: Plan generated but no steps executed.")

Goal: Find Experienced Python Developers with max salary 200000 and min 3 years experience, then draft a polite rejection email for them mentioning we will keep their resume.

--- FINAL RECRUITMENT EMAILS ---
Subject: Your Application for Our Python Developer Role

Dear [Candidate Name],

Thank you very much for your interest in the Python Developer position at [Company Name] and for the time you invested in our recruitment process.

After careful consideration, we have decided not to move forward with your application for this role at this time. This decision was difficult given the strength of your background, and it does not reflect negatively on your overall experience or abilities.

With your permission, we will keep your resume on file and may reach out should a future opportunity arise that more closely matches your profile and our current needs.

We appreciate your interest in [Company Name] and wish you every success in your ongoing career.

Best regards,  
[Your Name]  
[Your

In [16]:
OracleManager.close()